In [1]:
import os
import gzip
import bz2
import lzma
import subprocess
import shutil
import numpy as np
import pandas as pd
from datetime import datetime

# Additional compression libraries with proper aliased imports
SNAPPY_AVAILABLE = False
BROTLI_AVAILABLE = False
LZ4_AVAILABLE = False
ZSTD_AVAILABLE = False

try:
    import snappy as snappy_lib
    SNAPPY_AVAILABLE = True
except ImportError:
    print("Warning: python-snappy not available. Install with: pip install python-snappy")

try:
    import brotli as brotli_lib
    BROTLI_AVAILABLE = True
except ImportError:
    print("Warning: brotli not available. Install with: pip install brotli")

try:
    import lz4.frame as lz4_frame_lib
    import lz4.block as lz4_block_lib
    LZ4_AVAILABLE = True
except ImportError:
    print("Warning: lz4 not available. Install with: pip install lz4")

try:
    import zstandard as zstd_lib
    ZSTD_AVAILABLE = True
except ImportError:
    print("Warning: zstandard not available. Install with: pip install zstandard")


def check_command_exists(command):
    """Check if a command-line tool exists."""
    return shutil.which(command) is not None


# Check for command-line tools
LZOP_AVAILABLE = check_command_exists('lzop')
SEVEN_ZIP_AVAILABLE = check_command_exists('7z')
TSHARK_AVAILABLE = check_command_exists('tshark')

if not LZOP_AVAILABLE:
    print("Warning: lzop not available. Install with: sudo apt install lzop")
if not SEVEN_ZIP_AVAILABLE:
    print("Warning: 7z not available. Install with: sudo apt install p7zip-full")
if not TSHARK_AVAILABLE:
    print("Warning: tshark not available. Install with: sudo apt install tshark")


def compress_file(input_path, method):
    """Compress file using specified method and return compressed size in KB."""
    temp_output = input_path + ".compressed"

    try:
        if method == "gzip":
            with open(input_path, 'rb') as f_in, gzip.open(temp_output, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        elif method == "bzip2":
            with open(input_path, 'rb') as f_in, bz2.open(temp_output, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        elif method == "lzma":
            with open(input_path, 'rb') as f_in, lzma.open(temp_output, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        elif method == "ppmd":
            if not SEVEN_ZIP_AVAILABLE:
                return None
            temp_output = input_path + ".7z"
            command = ['7z', 'a', '-m0=PPMD', temp_output, input_path]
            subprocess.run(command, check=True, capture_output=True)
        
        elif method == "snappy":
            if not SNAPPY_AVAILABLE:
                return None
            with open(input_path, 'rb') as f_in:
                data = f_in.read()
            compressed = snappy_lib.compress(data)
            with open(temp_output, 'wb') as f_out:
                f_out.write(compressed)
        
        elif method == "brotli":
            if not BROTLI_AVAILABLE:
                return None
            with open(input_path, 'rb') as f_in:
                data = f_in.read()
            compressed = brotli_lib.compress(data)
            with open(temp_output, 'wb') as f_out:
                f_out.write(compressed)
        
        elif method == "lz4":
            if not LZ4_AVAILABLE:
                return None
            with open(input_path, 'rb') as f_in:
                data = f_in.read()
            compressed = lz4_frame_lib.compress(data)
            with open(temp_output, 'wb') as f_out:
                f_out.write(compressed)
        
        elif method == "lz4_raw":
            if not LZ4_AVAILABLE:
                return None
            with open(input_path, 'rb') as f_in:
                data = f_in.read()
            compressed = lz4_block_lib.compress(data)
            with open(temp_output, 'wb') as f_out:
                f_out.write(compressed)
        
        elif method == "zstd":
            if not ZSTD_AVAILABLE:
                return None
            with open(input_path, 'rb') as f_in:
                data = f_in.read()
            cctx = zstd_lib.ZstdCompressor()
            compressed = cctx.compress(data)
            with open(temp_output, 'wb') as f_out:
                f_out.write(compressed)
        
        elif method == "lzo":
            if not LZOP_AVAILABLE:
                return None
            temp_output = input_path + ".lzo"
            command = ['lzop', '-o', temp_output, input_path]
            subprocess.run(command, check=True, capture_output=True)
        
        else:
            return None

        size_kb = os.path.getsize(temp_output) / 1024
        os.remove(temp_output)
        return size_kb
    
    except Exception as e:
        # Clean up temp file if it exists
        if os.path.exists(temp_output):
            os.remove(temp_output)
        print(f"Warning: {method} compression failed for {input_path}: {e}")
        return None


def get_pcap_timestamps(file_path):
    """Extract start time, end time, and duration from pcap/pcapng file using tshark."""
    if not TSHARK_AVAILABLE:
        return None, None, None
    
    try:
        # Get first packet timestamp
        cmd_start = ['tshark', '-r', file_path, '-c', '1', '-T', 'fields', '-e', 'frame.time_epoch']
        result_start = subprocess.run(cmd_start, capture_output=True, text=True, check=True)
        start_epoch = float(result_start.stdout.strip())
        
        # Get last packet timestamp (read all packets, get the last one)
        cmd_end = ['tshark', '-r', file_path, '-T', 'fields', '-e', 'frame.time_epoch']
        result_end = subprocess.run(cmd_end, capture_output=True, text=True, check=True)
        timestamps = result_end.stdout.strip().split('\n')
        end_epoch = float(timestamps[-1]) if timestamps else start_epoch
        
        # Convert to human-readable format
        start_time = datetime.fromtimestamp(start_epoch).strftime('%Y-%m-%d %H:%M:%S.%f')
        end_time = datetime.fromtimestamp(end_epoch).strftime('%Y-%m-%d %H:%M:%S.%f')
        delta_seconds = end_epoch - start_epoch
        
        return start_time, end_time, delta_seconds
    
    except Exception as e:
        print(f"Warning: Could not extract timestamps from {file_path}: {e}")
        return None, None, None


def compute_entropy_min_size(file_path, bit_depth=8):
    """Compute theoretical minimum size based on Shannon entropy."""
    with open(file_path, 'rb') as f:
        data = list(f.read())
    hist = [data.count(i) for i in range(256)]
    total = sum(hist)
    ent = -np.sum([p * np.log2(p) for p in np.array(hist) / total if p > 0])
    original_size = os.path.getsize(file_path) / 1024
    min_size = (ent * original_size) / bit_depth
    return min_size


def compute_conditional_entropy(file_path):
    """Compute conditional entropy (first-order) minimum size."""
    with open(file_path, 'rb') as f:
        data = list(f.read())
    joint_counts = np.zeros((256, 256))
    for i in range(len(data) - 1):
        joint_counts[data[i], data[i + 1]] += 1
    joint_probs = joint_counts / np.sum(joint_counts)
    marginal_probs = np.sum(joint_probs, axis=1, keepdims=True)
    conditional_probs = np.divide(joint_probs, marginal_probs, where=marginal_probs != 0)
    conditional_entropy = -np.nansum(joint_probs * np.log2(conditional_probs, where=conditional_probs > 0))
    original_size = os.path.getsize(file_path) / 1024
    min_size = (conditional_entropy * original_size) / 8
    return min_size


def process_file(file_path):
    """Process a single pcap file and return all metrics."""
    print(f"Processing: {file_path}")
    
    # Original file size (KB)
    original_size = os.path.getsize(file_path) / 1024
    
    # Get pcap timestamps
    start_time, end_time, delta_time = get_pcap_timestamps(file_path)
    
    # Compression sizes
    gzip_size = compress_file(file_path, "gzip")
    bzip2_size = compress_file(file_path, "bzip2")
    lzma_size = compress_file(file_path, "lzma")
    ppmd_size = compress_file(file_path, "ppmd")
    snappy_size = compress_file(file_path, "snappy")
    brotli_size = compress_file(file_path, "brotli")
    lz4_size = compress_file(file_path, "lz4")
    lz4_raw_size = compress_file(file_path, "lz4_raw")
    zstd_size = compress_file(file_path, "zstd")
    lzo_size = compress_file(file_path, "lzo")
    
    # Entropy calculations
    entropy_8 = compute_entropy_min_size(file_path, 8)
    entropy_16 = compute_entropy_min_size(file_path, 16)
    entropy_conditional = compute_conditional_entropy(file_path)

    return {
        "File Name": os.path.basename(file_path),
        "Original Size (KB)": original_size,
        "Start Time": start_time,
        "End Time": end_time,
        "Delta Time (s)": delta_time,
        "GZIP Size (KB)": gzip_size,
        "BZIP2 Size (KB)": bzip2_size,
        "LZMA Size (KB)": lzma_size,
        "PPMD Size (KB)": ppmd_size,
        "SNAPPY Size (KB)": snappy_size,
        "BROTLI Size (KB)": brotli_size,
        "LZ4 Size (KB)": lz4_size,
        "LZ4_RAW Size (KB)": lz4_raw_size,
        "ZSTD Size (KB)": zstd_size,
        "LZO Size (KB)": lzo_size,
        "8-bit Entropy Min Size (KB)": entropy_8,
        "16-bit Entropy Min Size (KB)": entropy_16,
        "8-bit Conditional Entropy Min Size (KB)": entropy_conditional
    }


def main(dataset_dir, output_dir):
    """Main function to process all pcap files in the dataset."""
    os.makedirs(output_dir, exist_ok=True)

    categories = ["Idle", "Physical_Interaction", "Power", "Scenario", "Web_Interaction"]
    topologies = ["Topology_A", "Topology_B"]

    for category in categories:
        category_path = os.path.join(dataset_dir, category)
        if not os.path.exists(category_path):
            print(f"Category path not found: {category_path}")
            continue

        for topology in topologies:
            topology_path = os.path.join(category_path, topology)
            if not os.path.exists(topology_path):
                print(f"Topology path not found: {topology_path}")
                continue

            topology_results = []

            for root, _, files in os.walk(topology_path):
                for file in files:
                    if file.endswith(".pcapng"):
                        file_path = os.path.join(root, file)
                        result = process_file(file_path)
                        topology_results.append(result)

            if topology_results:
                final_df = pd.DataFrame(topology_results)
                
                # Reorder columns for better readability
                column_order = [
                    "File Name",
                    "Original Size (KB)",
                    "Start Time",
                    "End Time",
                    "Delta Time (s)",
                    "GZIP Size (KB)",
                    "BZIP2 Size (KB)",
                    "LZMA Size (KB)",
                    "PPMD Size (KB)",
                    "SNAPPY Size (KB)",
                    "BROTLI Size (KB)",
                    "LZ4 Size (KB)",
                    "LZ4_RAW Size (KB)",
                    "ZSTD Size (KB)",
                    "LZO Size (KB)",
                    "8-bit Entropy Min Size (KB)",
                    "16-bit Entropy Min Size (KB)",
                    "8-bit Conditional Entropy Min Size (KB)"
                ]
                final_df = final_df[column_order]
                
                output_file = os.path.join(output_dir, f"{category}_{topology}_results.csv")
                final_df.to_csv(output_file, index=False)
                print(f"Results saved to {output_file}")


if __name__ == "__main__":
    dataset_dir = "./Data"
    output_dir = "10-Loseless_size_results"
    main(dataset_dir, output_dir)

Processing: ./Data/Idle/Topology_A/idel1.pcapng
Processing: ./Data/Idle/Topology_A/idle3.pcapng
Processing: ./Data/Idle/Topology_A/idle2.pcapng
Results saved to 10-Loseless_size_results/Idle_Topology_A_results.csv
Processing: ./Data/Idle/Topology_B/idle3.pcapng
Processing: ./Data/Idle/Topology_B/idle1.pcapng
Processing: ./Data/Idle/Topology_B/idle2.pcapng
Results saved to 10-Loseless_size_results/Idle_Topology_B_results.csv
Processing: ./Data/Physical_Interaction/Topology_A/physicalInteraction1.pcapng
Processing: ./Data/Physical_Interaction/Topology_A/physicalInteraction2.pcapng
Processing: ./Data/Physical_Interaction/Topology_A/physicalInteraction3.pcapng
Results saved to 10-Loseless_size_results/Physical_Interaction_Topology_A_results.csv
Processing: ./Data/Physical_Interaction/Topology_B/physicalInteraction5.pcapng
Processing: ./Data/Physical_Interaction/Topology_B/physicalInteraction4.pcapng
Processing: ./Data/Physical_Interaction/Topology_B/physicalInteraction1.pcapng
Processing: 

/tmp/ipykernel_17342/438252009.py:204: RuntimeWarning: invalid value encountered in multiply
  conditional_entropy = -np.nansum(joint_probs * np.log2(conditional_probs, where=conditional_probs > 0))


Processing: ./Data/Power/Topology_A/powerMoesBulb1.pcapng
Processing: ./Data/Power/Topology_A/powerSmartPowerPlug1.pcapng
Processing: ./Data/Power/Topology_A/powerMoesBulb3.pcapng
Processing: ./Data/Power/Topology_A/powerPhilipsLamp2-1.pcapng
Processing: ./Data/Power/Topology_A/powerPhilipsLamp2-2.pcapng
Processing: ./Data/Power/Topology_A/powerMoesBulb2.pcapng
Processing: ./Data/Power/Topology_A/powerPhilipsLamp1-1.pcapng
Processing: ./Data/Power/Topology_A/powerSmartPowerPlug3.pcapng
Processing: ./Data/Power/Topology_A/powerSmartSocket2.pcapng
Processing: ./Data/Power/Topology_A/powerLedvanceBulb2.pcapng
Processing: ./Data/Power/Topology_A/powerLedvanceSmart+Plug3.pcapng
Processing: ./Data/Power/Topology_A/powerSmartPowerPlug5.pcapng
Processing: ./Data/Power/Topology_A/powerPhilipsLamp1-3.pcapng
Processing: ./Data/Power/Topology_A/powerLedvanceSmart+Plug1.pcapng
Processing: ./Data/Power/Topology_A/powerSmartPowerPlug4.pcapng
Processing: ./Data/Power/Topology_A/powerLedvancePlug2.pcap

/tmp/ipykernel_17342/438252009.py:204: RuntimeWarning: invalid value encountered in multiply
  conditional_entropy = -np.nansum(joint_probs * np.log2(conditional_probs, where=conditional_probs > 0))


Processing: ./Data/Power/Topology_B/powerMoesBulb1.pcapng


...

Processing: ./Data/Power/Topology_B/powerLedvanceZ3Plug2.pcapng


/tmp/ipykernel_17342/438252009.py:204: RuntimeWarning: invalid value encountered in multiply
  conditional_entropy = -np.nansum(joint_probs * np.log2(conditional_probs, where=conditional_probs > 0))


Results saved to 10-Loseless_size_results/Power_Topology_B_results.csv
Processing: ./Data/Scenario/Topology_A/scenario2.pcapng
Processing: ./Data/Scenario/Topology_A/scenario1.pcapng
Results saved to 10-Loseless_size_results/Scenario_Topology_A_results.csv
Processing: ./Data/Scenario/Topology_B/scenario1.pcapng


/tmp/ipykernel_17342/438252009.py:204: RuntimeWarning: invalid value encountered in multiply
  conditional_entropy = -np.nansum(joint_probs * np.log2(conditional_probs, where=conditional_probs > 0))


Results saved to 10-Loseless_size_results/Scenario_Topology_B_results.csv
Processing: ./Data/Web_Interaction/Topology_A/AppInteraction2.pcapng
Processing: ./Data/Web_Interaction/Topology_A/AppInteraction5.pcapng
Processing: ./Data/Web_Interaction/Topology_A/AppInteraction4.pcapng
Processing: ./Data/Web_Interaction/Topology_A/AppInteraction1.pcapng
Processing: ./Data/Web_Interaction/Topology_A/AppInteraction6.pcapng
Processing: ./Data/Web_Interaction/Topology_A/AppInteraction3.pcapng
Results saved to 10-Loseless_size_results/Web_Interaction_Topology_A_results.csv
Processing: ./Data/Web_Interaction/Topology_B/AppInteraction2.pcapng
Processing: ./Data/Web_Interaction/Topology_B/AppInteraction5.pcapng
Processing: ./Data/Web_Interaction/Topology_B/AppInteraction4.pcapng
Processing: ./Data/Web_Interaction/Topology_B/AppInteraction1.pcapng
Processing: ./Data/Web_Interaction/Topology_B/AppInteraction6.pcapng
Processing: ./Data/Web_Interaction/Topology_B/AppInteraction3.pcapng
Results saved to 

In [2]:
#!/usr/bin/env python3
"""
Merge and analyze compression results per category.

This script:
1. Loads all result CSV files from the compression analysis
2. Merges results per category (combining both topologies)
3. Computes total delta time and sums for each compression method
4. Creates comparison tables and visualizations
5. Generates detailed summary reports

Usage:
    python merge_and_analyze_results.py

Input directory: 10-Loseless_size_results/
Output directory: 11-Merged_results/
"""

import os
import pandas as pd
import numpy as np
from datetime import timedelta

def format_time(seconds):
    """Convert seconds to human-readable format."""
    return str(timedelta(seconds=seconds))


def merge_category_results(results_dir, output_dir):
    """
    Load all result CSV files and merge them per category.
    Compute total delta time and sums for each compression method.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Categories to process
    categories = ["Idle", "Physical_Interaction", "Power", "Scenario", "Web_Interaction"]
    
    # Columns that should be summed
    sum_columns = [
        "Original Size (KB)",
        "Delta Time (s)",
        "GZIP Size (KB)",
        "BZIP2 Size (KB)",
        "LZMA Size (KB)",
        "PPMD Size (KB)",
        "SNAPPY Size (KB)",
        "BROTLI Size (KB)",
        "LZ4 Size (KB)",
        "LZ4_RAW Size (KB)",
        "ZSTD Size (KB)",
        "LZO Size (KB)",
        "8-bit Entropy Min Size (KB)",
        "16-bit Entropy Min Size (KB)",
        "8-bit Conditional Entropy Min Size (KB)"
    ]
    
    all_results = []
    all_merged_dfs = {}
    
    print("=" * 80)
    print("MERGING RESULTS PER CATEGORY")
    print("=" * 80)
    
    for category in categories:
        print(f"\nProcessing category: {category}")
        print("-" * 80)
        
        # Find all CSV files for this category
        category_files = []
        for file in os.listdir(results_dir):
            if file.startswith(category) and file.endswith("_results.csv"):
                category_files.append(os.path.join(results_dir, file))
        
        if not category_files:
            print(f"  ⚠ No files found for category: {category}")
            continue
        
        # Load and concatenate all files for this category
        category_dfs = []
        for file_path in category_files:
            topology = "Topology_A" if "Topology_A" in file_path else "Topology_B"
            print(f"  ✓ Loading: {os.path.basename(file_path)} ({topology})")
            df = pd.read_csv(file_path)
            df['Topology'] = topology
            category_dfs.append(df)
        
        # Merge all dataframes for this category
        merged_df = pd.concat(category_dfs, ignore_index=True)
        all_merged_dfs[category] = merged_df
        
        # Save merged category file
        merged_output = os.path.join(output_dir, f"{category}_merged.csv")
        merged_df.to_csv(merged_output, index=False)
        print(f"  💾 Saved merged file: {merged_output}")
        print(f"  📊 Total files in category: {len(merged_df)}")
        
        # Compute summary statistics for this category
        summary = {"Category": category}
        summary["Number of Files"] = len(merged_df)
        
        for col in sum_columns:
            if col in merged_df.columns:
                # Remove None/NaN values before summing
                valid_values = merged_df[col].dropna()
                if len(valid_values) > 0:
                    summary[f"Total {col}"] = valid_values.sum()
                    summary[f"Mean {col}"] = valid_values.mean()
                    summary[f"Median {col}"] = valid_values.median()
                    summary[f"Std {col}"] = valid_values.std()
                    summary[f"Min {col}"] = valid_values.min()
                    summary[f"Max {col}"] = valid_values.max()
                else:
                    for stat in ["Total", "Mean", "Median", "Std", "Min", "Max"]:
                        summary[f"{stat} {col}"] = None
        
        all_results.append(summary)
        
        # Print summary for this category
        total_time = summary.get('Total Delta Time (s)', 0)
        total_size = summary.get('Total Original Size (KB)', 0)
        print(f"  ⏱  Total Delta Time: {format_time(total_time)} ({total_time:.2f} seconds)")
        print(f"  📦 Total Original Size: {total_size:.2f} KB ({total_size/1024:.2f} MB)")
    
    # Create overall summary DataFrame
    summary_df = pd.DataFrame(all_results)
    summary_output = os.path.join(output_dir, "category_summary.csv")
    summary_df.to_csv(summary_output, index=False)
    
    print(f"\n{'=' * 80}")
    print(f"✓ Summary saved to: {summary_output}")
    print(f"{'=' * 80}")
    
    return summary_df, all_merged_dfs


def create_detailed_report(summary_df, output_dir):
    """Create a detailed text report with all statistics."""
    
    report_file = os.path.join(output_dir, "detailed_report.txt")
    
    with open(report_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("COMPRESSION ANALYSIS - DETAILED REPORT\n")
        f.write("=" * 80 + "\n\n")
        
        for _, row in summary_df.iterrows():
            f.write(f"\nCategory: {row['Category']}\n")
            f.write("-" * 80 + "\n")
            f.write(f"Number of Files: {row['Number of Files']}\n")
            
            total_time = row.get('Total Delta Time (s)', 0)
            f.write(f"Total Delta Time: {format_time(total_time)} ({total_time:.2f} seconds)\n")
            
            original_size = row.get('Total Original Size (KB)', 0)
            f.write(f"Total Original Size: {original_size:.2f} KB ({original_size/1024:.2f} MB)\n")
            
            # Compression methods analysis
            f.write(f"\nCompression Results:\n")
            compression_methods = [
                ('GZIP', 'Total GZIP Size (KB)'),
                ('BZIP2', 'Total BZIP2 Size (KB)'),
                ('LZMA', 'Total LZMA Size (KB)'),
                ('PPMD', 'Total PPMD Size (KB)'),
                ('SNAPPY', 'Total SNAPPY Size (KB)'),
                ('BROTLI', 'Total BROTLI Size (KB)'),
                ('LZ4', 'Total LZ4 Size (KB)'),
                ('LZ4_RAW', 'Total LZ4_RAW Size (KB)'),
                ('ZSTD', 'Total ZSTD Size (KB)'),
                ('LZO', 'Total LZO Size (KB)')
            ]
            
            if original_size and original_size > 0:
                for method_name, col_name in compression_methods:
                    compressed_size = row.get(col_name)
                    if compressed_size and not pd.isna(compressed_size):
                        ratio = (compressed_size / original_size) * 100
                        savings = 100 - ratio
                        f.write(f"  {method_name:10s}: {compressed_size:10.2f} KB "
                               f"({ratio:6.2f}% | {savings:6.2f}% savings)\n")
            
            # Entropy analysis
            f.write(f"\nEntropy Analysis:\n")
            entropy_8 = row.get('Total 8-bit Entropy Min Size (KB)')
            entropy_16 = row.get('Total 16-bit Entropy Min Size (KB)')
            entropy_cond = row.get('Total 8-bit Conditional Entropy Min Size (KB)')
            
            if entropy_8 and not pd.isna(entropy_8):
                ratio_8 = (entropy_8 / original_size) * 100
                f.write(f"  8-bit Entropy:       {entropy_8:10.2f} KB ({ratio_8:6.2f}%)\n")
            if entropy_16 and not pd.isna(entropy_16):
                ratio_16 = (entropy_16 / original_size) * 100
                f.write(f"  16-bit Entropy:      {entropy_16:10.2f} KB ({ratio_16:6.2f}%)\n")
            if entropy_cond and not pd.isna(entropy_cond):
                ratio_cond = (entropy_cond / original_size) * 100
                f.write(f"  Conditional Entropy: {entropy_cond:10.2f} KB ({ratio_cond:6.2f}%)\n")
            
            f.write("\n")
    
    print(f"✓ Detailed report saved to: {report_file}")
    return report_file


def create_comparison_table(summary_df, output_dir):
    """Create a comparison table showing compression ratios across categories."""
    
    compression_methods = ['GZIP', 'BZIP2', 'LZMA', 'PPMD', 'SNAPPY', 'BROTLI', 
                          'LZ4', 'LZ4_RAW', 'ZSTD', 'LZO']
    
    comparison_data = []
    
    for _, row in summary_df.iterrows():
        category = row['Category']
        original_size = row.get('Total Original Size (KB)')
        
        if original_size and original_size > 0:
            row_data = {
                'Category': category,
                'Original Size (KB)': original_size,
                'Original Size (MB)': original_size / 1024,
                'Number of Files': row['Number of Files'],
                'Total Delta Time (s)': row.get('Total Delta Time (s)')
            }
            
            for method in compression_methods:
                col_name = f'Total {method} Size (KB)'
                compressed_size = row.get(col_name)
                if compressed_size and not pd.isna(compressed_size):
                    ratio = (compressed_size / original_size) * 100
                    savings = 100 - ratio
                    row_data[f'{method} Size (KB)'] = compressed_size
                    row_data[f'{method} Ratio (%)'] = ratio
                    row_data[f'{method} Savings (%)'] = savings
                else:
                    row_data[f'{method} Size (KB)'] = None
                    row_data[f'{method} Ratio (%)'] = None
                    row_data[f'{method} Savings (%)'] = None
            
            comparison_data.append(row_data)
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Save full comparison
    comparison_output = os.path.join(output_dir, "compression_comparison_full.csv")
    comparison_df.to_csv(comparison_output, index=False)
    print(f"✓ Full comparison saved to: {comparison_output}")
    
    # Create simplified comparison (just ratios)
    ratio_columns = ['Category'] + [f'{m} Ratio (%)' for m in compression_methods]
    ratio_df = comparison_df[ratio_columns]
    ratio_output = os.path.join(output_dir, "compression_ratios.csv")
    ratio_df.to_csv(ratio_output, index=False)
    print(f"✓ Compression ratios saved to: {ratio_output}")
    
    return comparison_df


def find_best_methods(summary_df, output_dir):
    """Identify the best compression method for each category."""
    
    compression_methods = ['GZIP', 'BZIP2', 'LZMA', 'PPMD', 'SNAPPY', 'BROTLI', 
                          'LZ4', 'LZ4_RAW', 'ZSTD', 'LZO']
    
    best_methods = []
    
    for _, row in summary_df.iterrows():
        category = row['Category']
        original_size = row.get('Total Original Size (KB)')
        
        if original_size and original_size > 0:
            method_ratios = {}
            
            for method in compression_methods:
                col_name = f'Total {method} Size (KB)'
                compressed_size = row.get(col_name)
                if compressed_size and not pd.isna(compressed_size):
                    ratio = (compressed_size / original_size) * 100
                    method_ratios[method] = {
                        'size': compressed_size,
                        'ratio': ratio,
                        'savings': 100 - ratio
                    }
            
            if method_ratios:
                best_method = min(method_ratios, key=lambda x: method_ratios[x]['ratio'])
                best_methods.append({
                    'Category': category,
                    'Best Method': best_method,
                    'Compressed Size (KB)': method_ratios[best_method]['size'],
                    'Compression Ratio (%)': method_ratios[best_method]['ratio'],
                    'Space Savings (%)': method_ratios[best_method]['savings'],
                    'Original Size (KB)': original_size
                })
    
    best_df = pd.DataFrame(best_methods)
    best_output = os.path.join(output_dir, "best_compression_methods.csv")
    best_df.to_csv(best_output, index=False)
    print(f"✓ Best methods saved to: {best_output}")
    
    # Print to console
    print("\n" + "=" * 80)
    print("BEST COMPRESSION METHOD PER CATEGORY")
    print("=" * 80)
    for _, row in best_df.iterrows():
        print(f"\n{row['Category']}:")
        print(f"  Best Method: {row['Best Method']}")
        print(f"  Compression Ratio: {row['Compression Ratio (%)']:.2f}%")
        print(f"  Space Savings: {row['Space Savings (%)']:.2f}%")
    
    return best_df



"""Main execution function."""

results_dir = "10-Loseless_size_results"
output_dir = "11-Merged_results"

# Check if results directory exists
if not os.path.exists(results_dir):
    print(f"❌ Error: Results directory '{results_dir}' not found.")
    print("Please ensure the compression analysis script has been run first.")

print("\n" + "=" * 80)
print("COMPRESSION RESULTS ANALYSIS")
print("=" * 80)
print(f"Input directory:  {results_dir}")
print(f"Output directory: {output_dir}")
print("=" * 80 + "\n")

# Merge results per category
summary_df, merged_dfs = merge_category_results(results_dir, output_dir)

# Create detailed report
create_detailed_report(summary_df, output_dir)

# Create comparison tables
create_comparison_table(summary_df, output_dir)

# Find best methods
find_best_methods(summary_df, output_dir)

print("\n" + "=" * 80)
print("✓ ALL PROCESSING COMPLETE!")
print("=" * 80)
print(f"\nOutput files in '{output_dir}':")
print("  - category_summary.csv              (Complete statistics per category)")
print("  - <Category>_merged.csv             (Merged data for each category)")
print("  - compression_comparison_full.csv   (Full comparison table)")
print("  - compression_ratios.csv            (Simplified ratio comparison)")
print("  - best_compression_methods.csv      (Best method per category)")
print("  - detailed_report.txt               (Human-readable report)")
print("=" * 80 + "\n")




COMPRESSION RESULTS ANALYSIS
Input directory:  10-Loseless_size_results
Output directory: 11-Merged_results

MERGING RESULTS PER CATEGORY

Processing category: Idle
--------------------------------------------------------------------------------
  ✓ Loading: Idle_Topology_A_results.csv (Topology_A)
  ✓ Loading: Idle_Topology_B_results.csv (Topology_B)
  💾 Saved merged file: 11-Merged_results/Idle_merged.csv
  📊 Total files in category: 6
  ⏱  Total Delta Time: 5:54:42.319745 (21282.32 seconds)
  📦 Total Original Size: 7137.56 KB (6.97 MB)

Processing category: Physical_Interaction
--------------------------------------------------------------------------------
  ✓ Loading: Physical_Interaction_Topology_B_results.csv (Topology_B)
  ✓ Loading: Physical_Interaction_Topology_A_results.csv (Topology_A)
  💾 Saved merged file: 11-Merged_results/Physical_Interaction_merged.csv
  📊 Total files in category: 10
  ⏱  Total Delta Time: 2:10:25.001420 (7825.00 seconds)
  📦 Total Original Size: 3206

In [1]:
#!/usr/bin/env python3
"""
Merge and analyze compression results per topology.

This script:
1. Loads all result CSV files from the compression analysis
2. Merges results per topology (Topology_A and Topology_B)
3. Computes total delta time and sums for each compression method
4. Creates comparison tables and visualizations
5. Generates detailed summary reports

Usage:
    python merge_and_analyze_results_by_topology.py

Input directory: 10-Loseless_size_results/
Output directory: 11-Merged_results_by_topology/
"""

import os
import pandas as pd
import numpy as np
from datetime import timedelta

def format_time(seconds):
    """Convert seconds to human-readable format."""
    return str(timedelta(seconds=seconds))


def merge_topology_results(results_dir, output_dir):
    """
    Load all result CSV files and merge them per topology.
    Compute total delta time and sums for each compression method.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Topologies to process
    topologies = ["Topology_A", "Topology_B"]
    
    # Columns that should be summed
    sum_columns = [
        "Original Size (KB)",
        "Delta Time (s)",
        "GZIP Size (KB)",
        "BZIP2 Size (KB)",
        "LZMA Size (KB)",
        "PPMD Size (KB)",
        "SNAPPY Size (KB)",
        "BROTLI Size (KB)",
        "LZ4 Size (KB)",
        "LZ4_RAW Size (KB)",
        "ZSTD Size (KB)",
        "LZO Size (KB)",
        "8-bit Entropy Min Size (KB)",
        "16-bit Entropy Min Size (KB)",
        "8-bit Conditional Entropy Min Size (KB)"
    ]
    
    all_results = []
    all_merged_dfs = {}
    
    print("=" * 80)
    print("MERGING RESULTS PER TOPOLOGY")
    print("=" * 80)
    
    for topology in topologies:
        print(f"\nProcessing topology: {topology}")
        print("-" * 80)
        
        # Find all CSV files for this topology
        topology_files = []
        for file in os.listdir(results_dir):
            if topology in file and file.endswith("_results.csv"):
                topology_files.append(os.path.join(results_dir, file))
        
        if not topology_files:
            print(f"  ⚠ No files found for topology: {topology}")
            continue
        
        # Load and concatenate all files for this topology
        topology_dfs = []
        for file_path in topology_files:
            # Extract category from filename
            filename = os.path.basename(file_path)
            # Remove topology and _results.csv to get category
            category = filename.replace(f"_{topology}_results.csv", "")
            print(f"  ✓ Loading: {os.path.basename(file_path)} (Category: {category})")
            df = pd.read_csv(file_path)
            df['Category'] = category
            topology_dfs.append(df)
        
        # Merge all dataframes for this topology
        merged_df = pd.concat(topology_dfs, ignore_index=True)
        all_merged_dfs[topology] = merged_df
        
        # Save merged topology file
        merged_output = os.path.join(output_dir, f"{topology}_merged.csv")
        merged_df.to_csv(merged_output, index=False)
        print(f"  💾 Saved merged file: {merged_output}")
        print(f"  📊 Total files in topology: {len(merged_df)}")
        
        # Compute summary statistics for this topology
        summary = {"Topology": topology}
        summary["Number of Files"] = len(merged_df)
        
        for col in sum_columns:
            if col in merged_df.columns:
                # Remove None/NaN values before summing
                valid_values = merged_df[col].dropna()
                if len(valid_values) > 0:
                    summary[f"Total {col}"] = valid_values.sum()
                    summary[f"Mean {col}"] = valid_values.mean()
                    summary[f"Median {col}"] = valid_values.median()
                    summary[f"Std {col}"] = valid_values.std()
                    summary[f"Min {col}"] = valid_values.min()
                    summary[f"Max {col}"] = valid_values.max()
                else:
                    for stat in ["Total", "Mean", "Median", "Std", "Min", "Max"]:
                        summary[f"{stat} {col}"] = None
        
        all_results.append(summary)
        
        # Print summary for this topology
        total_time = summary.get('Total Delta Time (s)', 0)
        total_size = summary.get('Total Original Size (KB)', 0)
        print(f"  ⏱  Total Delta Time: {format_time(total_time)} ({total_time:.2f} seconds)")
        print(f"  📦 Total Original Size: {total_size:.2f} KB ({total_size/1024:.2f} MB)")
    
    # Create overall summary DataFrame
    summary_df = pd.DataFrame(all_results)
    summary_output = os.path.join(output_dir, "topology_summary.csv")
    summary_df.to_csv(summary_output, index=False)
    
    print(f"\n{'=' * 80}")
    print(f"✓ Summary saved to: {summary_output}")
    print(f"{'=' * 80}")
    
    return summary_df, all_merged_dfs


def create_detailed_report(summary_df, output_dir):
    """Create a detailed text report with all statistics."""
    
    report_file = os.path.join(output_dir, "detailed_report.txt")
    
    with open(report_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("COMPRESSION ANALYSIS - DETAILED REPORT (BY TOPOLOGY)\n")
        f.write("=" * 80 + "\n\n")
        
        for _, row in summary_df.iterrows():
            f.write(f"\nTopology: {row['Topology']}\n")
            f.write("-" * 80 + "\n")
            f.write(f"Number of Files: {row['Number of Files']}\n")
            
            total_time = row.get('Total Delta Time (s)', 0)
            f.write(f"Total Delta Time: {format_time(total_time)} ({total_time:.2f} seconds)\n")
            
            original_size = row.get('Total Original Size (KB)', 0)
            f.write(f"Total Original Size: {original_size:.2f} KB ({original_size/1024:.2f} MB)\n")
            
            # Compression methods analysis
            f.write(f"\nCompression Results:\n")
            compression_methods = [
                ('GZIP', 'Total GZIP Size (KB)'),
                ('BZIP2', 'Total BZIP2 Size (KB)'),
                ('LZMA', 'Total LZMA Size (KB)'),
                ('PPMD', 'Total PPMD Size (KB)'),
                ('SNAPPY', 'Total SNAPPY Size (KB)'),
                ('BROTLI', 'Total BROTLI Size (KB)'),
                ('LZ4', 'Total LZ4 Size (KB)'),
                ('LZ4_RAW', 'Total LZ4_RAW Size (KB)'),
                ('ZSTD', 'Total ZSTD Size (KB)'),
                ('LZO', 'Total LZO Size (KB)')
            ]
            
            if original_size and original_size > 0:
                for method_name, col_name in compression_methods:
                    compressed_size = row.get(col_name)
                    if compressed_size and not pd.isna(compressed_size):
                        ratio = (compressed_size / original_size) * 100
                        savings = 100 - ratio
                        f.write(f"  {method_name:10s}: {compressed_size:10.2f} KB "
                               f"({ratio:6.2f}% | {savings:6.2f}% savings)\n")
            
            # Entropy analysis
            f.write(f"\nEntropy Analysis:\n")
            entropy_8 = row.get('Total 8-bit Entropy Min Size (KB)')
            entropy_16 = row.get('Total 16-bit Entropy Min Size (KB)')
            entropy_cond = row.get('Total 8-bit Conditional Entropy Min Size (KB)')
            
            if entropy_8 and not pd.isna(entropy_8):
                ratio_8 = (entropy_8 / original_size) * 100
                f.write(f"  8-bit Entropy:       {entropy_8:10.2f} KB ({ratio_8:6.2f}%)\n")
            if entropy_16 and not pd.isna(entropy_16):
                ratio_16 = (entropy_16 / original_size) * 100
                f.write(f"  16-bit Entropy:      {entropy_16:10.2f} KB ({ratio_16:6.2f}%)\n")
            if entropy_cond and not pd.isna(entropy_cond):
                ratio_cond = (entropy_cond / original_size) * 100
                f.write(f"  Conditional Entropy: {entropy_cond:10.2f} KB ({ratio_cond:6.2f}%)\n")
            
            f.write("\n")
    
    print(f"✓ Detailed report saved to: {report_file}")
    return report_file


def create_comparison_table(summary_df, output_dir):
    """Create a comparison table showing compression ratios across topologies."""
    
    compression_methods = ['GZIP', 'BZIP2', 'LZMA', 'PPMD', 'SNAPPY', 'BROTLI', 
                          'LZ4', 'LZ4_RAW', 'ZSTD', 'LZO']
    
    comparison_data = []
    
    for _, row in summary_df.iterrows():
        topology = row['Topology']
        original_size = row.get('Total Original Size (KB)')
        
        if original_size and original_size > 0:
            row_data = {
                'Topology': topology,
                'Original Size (KB)': original_size,
                'Original Size (MB)': original_size / 1024,
                'Number of Files': row['Number of Files'],
                'Total Delta Time (s)': row.get('Total Delta Time (s)')
            }
            
            for method in compression_methods:
                col_name = f'Total {method} Size (KB)'
                compressed_size = row.get(col_name)
                if compressed_size and not pd.isna(compressed_size):
                    ratio = (compressed_size / original_size) * 100
                    savings = 100 - ratio
                    row_data[f'{method} Size (KB)'] = compressed_size
                    row_data[f'{method} Ratio (%)'] = ratio
                    row_data[f'{method} Savings (%)'] = savings
                else:
                    row_data[f'{method} Size (KB)'] = None
                    row_data[f'{method} Ratio (%)'] = None
                    row_data[f'{method} Savings (%)'] = None
            
            comparison_data.append(row_data)
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Save full comparison
    comparison_output = os.path.join(output_dir, "compression_comparison_full.csv")
    comparison_df.to_csv(comparison_output, index=False)
    print(f"✓ Full comparison saved to: {comparison_output}")
    
    # Create simplified comparison (just ratios)
    ratio_columns = ['Topology'] + [f'{m} Ratio (%)' for m in compression_methods]
    ratio_df = comparison_df[ratio_columns]
    ratio_output = os.path.join(output_dir, "compression_ratios.csv")
    ratio_df.to_csv(ratio_output, index=False)
    print(f"✓ Compression ratios saved to: {ratio_output}")
    
    return comparison_df


def find_best_methods(summary_df, output_dir):
    """Identify the best compression method for each topology."""
    
    compression_methods = ['GZIP', 'BZIP2', 'LZMA', 'PPMD', 'SNAPPY', 'BROTLI', 
                          'LZ4', 'LZ4_RAW', 'ZSTD', 'LZO']
    
    best_methods = []
    
    for _, row in summary_df.iterrows():
        topology = row['Topology']
        original_size = row.get('Total Original Size (KB)')
        
        if original_size and original_size > 0:
            method_ratios = {}
            
            for method in compression_methods:
                col_name = f'Total {method} Size (KB)'
                compressed_size = row.get(col_name)
                if compressed_size and not pd.isna(compressed_size):
                    ratio = (compressed_size / original_size) * 100
                    method_ratios[method] = {
                        'size': compressed_size,
                        'ratio': ratio,
                        'savings': 100 - ratio
                    }
            
            if method_ratios:
                best_method = min(method_ratios, key=lambda x: method_ratios[x]['ratio'])
                best_methods.append({
                    'Topology': topology,
                    'Best Method': best_method,
                    'Compressed Size (KB)': method_ratios[best_method]['size'],
                    'Compression Ratio (%)': method_ratios[best_method]['ratio'],
                    'Space Savings (%)': method_ratios[best_method]['savings'],
                    'Original Size (KB)': original_size
                })
    
    best_df = pd.DataFrame(best_methods)
    best_output = os.path.join(output_dir, "best_compression_methods.csv")
    best_df.to_csv(best_output, index=False)
    print(f"✓ Best methods saved to: {best_output}")
    
    # Print to console
    print("\n" + "=" * 80)
    print("BEST COMPRESSION METHOD PER TOPOLOGY")
    print("=" * 80)
    for _, row in best_df.iterrows():
        print(f"\n{row['Topology']}:")
        print(f"  Best Method: {row['Best Method']}")
        print(f"  Compression Ratio: {row['Compression Ratio (%)']:.2f}%")
        print(f"  Space Savings: {row['Space Savings (%)']:.2f}%")
    
    return best_df


if __name__ == "__main__":
    """Main execution function."""
    
    results_dir = "10-Loseless_size_results"
    output_dir = "11-Merged_results_by_topology"
    
    # Check if results directory exists
    if not os.path.exists(results_dir):
        print(f"❌ Error: Results directory '{results_dir}' not found.")
        print("Please ensure the compression analysis script has been run first.")
        exit(1)
    
    print("\n" + "=" * 80)
    print("COMPRESSION RESULTS ANALYSIS (BY TOPOLOGY)")
    print("=" * 80)
    print(f"Input directory:  {results_dir}")
    print(f"Output directory: {output_dir}")
    print("=" * 80 + "\n")
    
    # Merge results per topology
    summary_df, merged_dfs = merge_topology_results(results_dir, output_dir)
    
    # Create detailed report
    create_detailed_report(summary_df, output_dir)
    
    # Create comparison tables
    create_comparison_table(summary_df, output_dir)
    
    # Find best methods
    find_best_methods(summary_df, output_dir)
    
    print("\n" + "=" * 80)
    print("✓ ALL PROCESSING COMPLETE!")
    print("=" * 80)
    print(f"\nOutput files in '{output_dir}':")
    print("  - topology_summary.csv              (Complete statistics per topology)")
    print("  - <Topology>_merged.csv             (Merged data for each topology)")
    print("  - compression_comparison_full.csv   (Full comparison table)")
    print("  - compression_ratios.csv            (Simplified ratio comparison)")
    print("  - best_compression_methods.csv      (Best method per topology)")
    print("  - detailed_report.txt               (Human-readable report)")
    print("=" * 80 + "\n")


COMPRESSION RESULTS ANALYSIS (BY TOPOLOGY)
Input directory:  10-Loseless_size_results
Output directory: 11-Merged_results_by_topology

MERGING RESULTS PER TOPOLOGY

Processing topology: Topology_A
--------------------------------------------------------------------------------
  ✓ Loading: Physical_Interaction_Topology_A_results.csv (Category: Physical_Interaction)
  ✓ Loading: Web_Interaction_Topology_A_results.csv (Category: Web_Interaction)
  ✓ Loading: Idle_Topology_A_results.csv (Category: Idle)
  ✓ Loading: Power_Topology_A_results.csv (Category: Power)
  ✓ Loading: Scenario_Topology_A_results.csv (Category: Scenario)
  💾 Saved merged file: 11-Merged_results_by_topology/Topology_A_merged.csv
  📊 Total files in topology: 44
  ⏱  Total Delta Time: 6:49:17.419477 (24557.42 seconds)
  📦 Total Original Size: 6692.56 KB (6.54 MB)

Processing topology: Topology_B
--------------------------------------------------------------------------------
  ✓ Loading: Physical_Interaction_Topology_

In [8]:
import os
import pandas as pd
import numpy as np

def process_files(folder_path):
    merged_a = None
    merged_b = None

    for file in os.listdir(folder_path):
        if file.endswith('.csv'):
            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)

            for col in df.columns:
                if col != 'File Name':
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            if '_Topology_A_' in file:
                merged_a = df if merged_a is None else pd.concat([merged_a, df], axis=0, ignore_index=True)
            elif '_Topology_B_' in file:
                merged_b = df if merged_b is None else pd.concat([merged_b, df], axis=0, ignore_index=True)

    if merged_a is not None:
        process_single_topology(merged_a, 'Topology_A')

    if merged_b is not None:
        process_single_topology(merged_b, 'Topology_B')


def process_single_topology(df, topology_name):
    if 'Original Size (KB)' not in df.columns:
        return

    original_size = df['Original Size (KB)']
    result_df = df.copy()

    for col in df.columns:
        if col not in ['Original Size (KB)', 'File Name']:
            result_df[f'{col}_ratio'] = original_size/df[col]
    ratio_columns = [col for col in result_df.columns if col.endswith('_ratio')]

    stats = pd.DataFrame({
        'Column': ratio_columns,
        'Mean': [result_df[col].mean() for col in ratio_columns],
        'Variance': [result_df[col].var() for col in ratio_columns]
    })

    mean_row = {col: stats.loc[stats['Column'] == col, 'Mean'].values[0] if col in ratio_columns else
                'Mean' if col == 'file name' else np.nan
                for col in result_df.columns}

    var_row = {col: stats.loc[stats['Column'] == col, 'Variance'].values[0] if col in ratio_columns else
               'Variance' if col == 'file name' else np.nan
               for col in result_df.columns}

    result_df = pd.concat([
        result_df,
        pd.DataFrame([mean_row]),
        pd.DataFrame([var_row])
    ], ignore_index=True)

    output_file = f'merged_{topology_name}_with_ratios.csv'
    result_df.to_csv(output_file, index=False)
    print(f"Saved {topology_name} to {output_file}")

folder_path = "10-Loseless_size_results"
process_files(folder_path)


Saved Topology_A to merged_Topology_A_with_ratios.csv
Saved Topology_B to merged_Topology_B_with_ratios.csv
